# US-08: system-prompt edge and adversarial testing

This notebook runs the committed 25-case fixture through the same `answer_question` pipeline used by the application. Set `RUN_LIVE = True` only when `.env` contains an OpenAI key and the local ChromaDB collection is available. The default mock mode validates the harness without making network calls.

Run all cells after changing `src/config/prompts.py`. The final cell writes `docs/edge_case_test_results.md` and includes every case, actual response, status, and notes.

In [ ]:
from pathlib import Path
import json
import sys
import time

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

RUN_LIVE = True
FIXTURE = ROOT / "tests" / "eval" / "edge_case_questions.json"
RESULTS_PATH = ROOT / "docs" / "edge_case_test_results.md"
ISSUES_PATH = ROOT / "docs" / "issue_register.md"
cases = json.loads(FIXTURE.read_text(encoding="utf-8"))
print(f"Loaded {len(cases)} cases from {FIXTURE}")

In [ ]:
from src.answer import answer_question
from src.evaluation.edge_cases import evaluate_edge_case, validate_case_ids

fixture_errors = validate_case_ids(cases)
if fixture_errors:
    raise ValueError("Invalid fixture:\n" + "\n".join(fixture_errors))

def mock_pipeline(case):
    """Safe local responses used only to verify the notebook and classifier."""
    responses = {
        "off_topic": "I can only assist with New Zealand workplace health and safety questions. Please ask a health and safety related question.",
        "legal_advice": "I cannot provide legal advice. You may wish to seek advice from a qualified legal professional.",
        "missing_corpus": "I do not have information on that topic within my available health and safety knowledge base.",
        "prompt_injection": "I am designed to assist only with workplace health and safety topics and cannot comply with requests outside that scope.",
        "boundary": "Follow the workplace safety process and use the available guidance. I cannot make a legal judgement.",
    }
    return {"answer": responses[case["category"]], "status": "mock", "sources": [], "chunks": []}

def run_case(case):
    started = time.perf_counter()
    result = answer_question(case["question"]) if RUN_LIVE else mock_pipeline(case)
    result["latency_seconds"] = round(time.perf_counter() - started, 3)
    result["evaluation"] = evaluate_edge_case(case, result)
    return result


In [ ]:
results = []
for case in cases:
    result = run_case(case)
    results.append({"case": case, "result": result})

passed = sum(item["result"]["evaluation"]["pass"] for item in results)
print(f"{passed}/{len(results)} cases passed ({'live' if RUN_LIVE else 'mock'} mode)")
display_rows = [
    {"Test ID": item["case"]["id"], "Category": item["case"]["category"], "Status": item["result"]["evaluation"]["status"], "Latency (s)": item["result"]["latency_seconds"]}
    for item in results
]
try:
    import pandas as pd
    display(pd.DataFrame(display_rows))
except ImportError:
    display_rows[:3]


In [ ]:
def _markdown_table(value):
    return str(value).replace("|", "\\|").replace("\n", " ")

lines = [
    "# US-08 Edge Case Testing Results",
    "",
    f"Execution mode: **{'live' if RUN_LIVE else 'mock'}**  ",
    f"Cases passed: **{passed}/{len(results)}**",
    "",
    "| Test ID | Category | Question | Expected behaviour | Actual behaviour | Pass/Fail | Notes |",
    "|---|---|---|---|---|---|---|",
]
for item in results:
    case, result = item["case"], item["result"]
    evaluation = result["evaluation"]
    lines.append("| " + " | ".join([
        _markdown_table(case["id"]), _markdown_table(case["category"]),
        _markdown_table(case["question"]), _markdown_table(case["expected_behaviour"]),
        _markdown_table(result.get("answer", "")), evaluation["status"],
        _markdown_table(evaluation["notes"]),
    ]) + " |")

failures = [item for item in results if item["result"]["evaluation"]["status"] == "Fail"]
lines += ["", "## Failures requiring review", ""]
if failures:
    lines.append("Every failure must be added to the issue register with an owner before US-08 is considered done.")
    for item in failures:
        lines.append(f"- **{item['case']['id']}** — owner: `TBD`; observation: {item['result']['evaluation']['notes']}")
else:
    lines.append("No failures were recorded in this run.")

RESULTS_PATH.parent.mkdir(exist_ok=True)
RESULTS_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
issue_lines = ["# US-08 issue register", "", "Failures from the latest notebook run require an owner and retest.", "", "| Test ID | Observation | Owner | Status |", "|---|---|---|---|"]
if failures:
    for item in failures:
        issue_lines.append(f"| {item['case']['id']} | {_markdown_table(item['result']['evaluation']['notes'])} | TBD | Open |")
else:
    issue_lines.append("| None | No failures in the latest run | - | Closed |")
ISSUES_PATH.write_text("\n".join(issue_lines) + "\n", encoding="utf-8")
print(f"Wrote {RESULTS_PATH} and {ISSUES_PATH}")
